# L4b: Breadth-First and Depth-First Search

__Which vertices can we reach, and in what order will we visit them?__ Breadth-first search (BFS) and depth-first search (DFS) answer these questions by following graph edges. From the same starting vertex, they explore the same reachable vertices in different ways: breadth-first search proceeds in layers, while depth-first search follows one branch before returning to another.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * **Build and validate an adjacency list:** Use the supplied code to construct and check outgoing-neighbor lists for a directed graph, and explain why the searches ignore edge weights. Relate the dictionary keys and neighbor lists to the graph, including vertices with no outgoing edges.
> * **Implement DFS and BFS:** Complete recursive depth-first search and queue-based breadth-first search, using visited sets to prevent repeated exploration. Explain how recursive calls and a first-in, first-out queue determine the exploration order, including when vertices must be marked as visited.
> * **Compare traversal results:** Explain how edge directions and the starting vertex affect reachability, and distinguish traversal order from a directed path. Use the computed results to explain why the algorithms can reach the same vertices in different orders.

We will work with the directed graph introduced in [L4a](../L4a/CHEME-5800-L4a-Lecture-GraphAndTreeRepresentations-Fall-2026.ipynb), using its adjacency list to guide both searches.

Let's get started!

___


## Algorithms

Both algorithms need to remember where to continue the search. Our depth-first implementation uses the active recursive calls, while our breadth-first implementation uses a queue.

* __Depth-first search__ follows one branch recursively. A call explores a neighbor and waits for that neighbor's call to finish before considering the next neighbor. Returning from a completed call is how the search resumes at an earlier vertex. [Read the DFS algorithm notebook](CHEME-5800-L4b-Algorithm-DepthFirstSearch-Fall-2026.ipynb).
* __Breadth-first search__ uses a first-in, first-out queue: newly discovered vertices join the back, while processing begins at the front. This makes the search explore vertices in layers, where each layer corresponds to the minimum number of edges needed to reach its vertices from the start. [Read the BFS algorithm notebook](CHEME-5800-L4b-Algorithm-BreadthFirstSearch-Fall-2026.ipynb).

Both searches use a visited set to avoid repeated exploration when edges converge or form a cycle. Their different ways of continuing the search can produce different visit orders even when the graph, starting vertex, and neighbor order are the same.

___


## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package and the `Test` standard library used to check our calculations.

Let's set up our code environment:


In [ ]:
# Rerun after saving src/Compute.jl so later calls use your edited functions.
include(joinpath(@__DIR__, "Include.jl")); # load the environment and student functions

See the [Julia documentation](https://docs.julialang.org/en/v1/) for language details and the [`Test` documentation](https://docs.julialang.org/en/v1/stdlib/Test/) for the checks used here. The student traversal functions are defined in [`src/Compute.jl`](src/Compute.jl).

The setup also loads [`DataFrames.jl`](https://dataframes.juliadata.org/stable/) for tables and [`PrettyTables.jl`](https://ronisbr.github.io/PrettyTables.jl/stable/) for their display.

After saving your changes to [`src/Compute.jl`](src/Compute.jl) in Task 2, rerun the setup cell to load the updated traversal functions. You do not need to restart the kernel.

___


## Task 1: Build and validate the directed graph

In this task, we will build and validate the adjacency list for both traversals using the supplied code. The file [`data/SimpleGraph.txt`](data/SimpleGraph.txt) describes the six-vertex directed graph shown below. Each record contains a source vertex, a target vertex, and an edge weight. We first load these records into the course graph model, then extract the outgoing neighbors of each vertex.

<div>
    <center>
        <img src="figs/Fig-Example-Graph.svg" width="680" alt="Six-vertex directed graph with seven weighted edges used to compare depth-first and breadth-first traversal"/>
    </center>
</div>

__What information does a traversal use?__ Both algorithms follow outgoing edges to find reachable vertices. The adjacency list maps each vertex to its outgoing neighbors. For example, the edges `1 → 2` and `1 → 3` give the entry `1 => [2, 3]`, which tells either search where it can go directly from vertex 1.

Edge weights do not affect this reachability calculation, so our adjacency list stores only the vertex identifiers. The graph model retains the weights.

We supply [the `parse_edge_record(...)` function](docs/traversal-functions.md#parse_edge_record) to read the three fields from each data record. The file reader skips comments before passing records to this function:

In [ ]:
"""
    parse_edge_record(record::String, delimiter::Char = ',')

Parse `source,target,weight` text into the tuple expected by
`MyGraphEdgeModels(...)`. Return `nothing` if the field count is not three;
invalid numeric fields raise a parsing error.
"""
function parse_edge_record(record::String, delimiter::Char = ',')
    fields = strip.(split(record, delimiter)); # trim whitespace around each field
    # The reader expects a tuple; nothing causes a loading error, not a skipped record.
    length(fields) == 3 || return nothing

    source = parse(Int, fields[1]);       # directed-edge source identifier
    target = parse(Int, fields[2]);       # directed-edge target identifier
    weight = parse(Float64, fields[3]);   # edge cost (arbitrary units); unused by the searches
    return (source, target, weight)
end

We load the edge records with [the `MyGraphEdgeModels(...)` function](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.MyGraphEdgeModels-Tuple%7BString%2C%20Function%7D) and construct the course graph model. We then pass the source-target pairs to [the `adjacency_from_edges(...)` function](docs/traversal-functions.md#adjacency_from_edges) to build the adjacency list.

Each vertex appearing in the edge file receives an entry, including vertices with no outgoing edges. The helper removes duplicate edges and sorts each vertex's outgoing neighbors.


In [ ]:
# Load weighted edges -
# The reader skips comments and empty lines, then calls our parser for each data record.
edge_file = joinpath(CHEME5800_L4B_DATA, "SimpleGraph.txt");
edge_models = MyGraphEdgeModels(edge_file, parse_edge_record; delim = ',', comment = '#');

# Build the weighted graph -
directed_graph = build(MySimpleDirectedGraphModel, edge_models);

# Extract connectivity for the searches -
# Dictionary keys identify edges; traversal needs the source and target vertex IDs.
edge_pairs = [(edge.source, edge.target) for edge in values(edge_models)]; # omit weights
adjacency = L4bTraversal.adjacency_from_edges(edge_pairs); # remove duplicates and sort neighbors

Before displaying the table, use the figure to predict the outgoing neighbors of vertices 2, 5, and 6. The table lists each vertex and its outgoing neighbors; the symbol `∅` represents an empty neighbor list.



In [ ]:
# Display outgoing neighbors -
# Convert lists to text only in this table; adjacency keeps its integer vectors.
vertices = sort!(collect(keys(adjacency))); # fixed row order for comparison with the figure
adjacency_table = DataFrame(
    vertex = vertices,
    # Display an empty neighbor list as ∅.
    outgoing_neighbors = [isempty(adjacency[v]) ? "∅" : join(adjacency[v], ", ") for v in vertices],
);
pretty_table(adjacency_table)

__Does the representation match the graph?__ We check the six vertices, seven edges, and outgoing-neighbor list for each vertex.


In [ ]:
@testset "L4b directed-graph representation" begin
    # Check graph size and vertex identifiers -
    @test length(directed_graph.nodes) == 6
    @test length(directed_graph.edges) == 7
    @test vertices == collect(1:6)

    # Counts alone cannot detect reversed edges; check every outgoing-neighbor list.
    @test adjacency == Dict(
        1 => [2, 3],
        2 => [3, 4],
        3 => [5],
        4 => [6],
        5 => [4],
        6 => Int64[], # a reachable vertex must still be represented when it has no outgoing edges
    )
end;

We now have a checked adjacency list for the directed graph. Next, we will use it to implement the two searches and compare their visit orders.

___


## Task 2: Implement DFS and BFS

In this task, we will complete DFS and BFS in two stages, checking each traversal from vertex 1 before comparing their results.

The [`depth_first_order(...)`](docs/traversal-functions.md#depth_first_order) and [`breadth_first_order(...)`](docs/traversal-functions.md#breadth_first_order) functions are defined in [`src/Compute.jl`](src/Compute.jl). Both accept the same inputs and must meet the following requirements.

> __Traversal inputs and results:__
>
> __Inputs__
>
> * `adjacency::AbstractDict`: maps every vertex identifier to its outgoing neighbors. Neither function may mutate this dictionary or its neighbor collections.
> * `start::Integer`: the vertex at which traversal begins. A Boolean is not a valid identifier even though `Bool <: Integer` in Julia.
>
> __Output__
>
> * `Vector{Int64}`: every vertex reachable from `start`, recorded exactly once in traversal order. Examine outgoing neighbors in ascending identifier order so the result is reproducible.
>
> __Errors__
>
> * [`ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError): `start` is a Boolean or does not appear as a key in `adjacency`.

An unfinished traversal raises an error identifying its remaining TODOs.


### Complete and check DFS

Use the [DFS algorithm notebook](CHEME-5800-L4b-Algorithm-DepthFirstSearch-Fall-2026.ipynb) to complete **TODOs 1–3** in [`src/Compute.jl`](src/Compute.jl):

1. Validate the starting vertex and create an empty visited set and result vector.
2. Define the recursive helper: skip visited vertices; otherwise record the vertex and recursively explore its outgoing neighbors in ascending order.
3. Call the helper on the starting vertex and return the DFS order.

Save your changes, rerun the setup cell, and run the DFS cell below:


In [ ]:
# Compute depth-first visit order -
dfs_order = L4bTraversal.depth_first_order(adjacency, 1);
dfs_order # entry k is the kth vertex visited, not necessarily vertex k

Starting at vertex 1, the search follows `1 → 2 → 3 → 5 → 4 → 6`. It visits all six vertices before backtracking begins, giving the order `[1, 2, 3, 5, 4, 6]`. Check your result against this order before moving on to BFS.


### Complete and check BFS

For breadth-first search, we will use the [`MyQueue`](../../../code/src/StacksQueues.jl) type introduced in [L3b](../../week-03/L3b/CHEME-5800-L3b-Lecture-StacksAndQueues-Fall-2026.ipynb) to hold vertices waiting to be explored. Its first-in, first-out behavior ensures that vertices are processed in the order they are discovered.

Use the [BFS algorithm notebook](CHEME-5800-L4b-Algorithm-BreadthFirstSearch-Fall-2026.ipynb) to complete **TODOs 4–6** in [`src/Compute.jl`](src/Compute.jl). The queue processes vertices one layer at a time.

4. Validate the starting vertex, initialize its visited set, result vector, and queue, then mark and enqueue the starting vertex.
5. While the queue is not empty, remove the vertex at the front and append it to the result vector.
6. Examine outgoing neighbors in ascending order; mark and enqueue each unvisited neighbor.

Save your changes and rerun the setup cell to load the completed BFS function. Then run the cell below:


In [ ]:
# Compute breadth-first visit order -
bfs_order = L4bTraversal.breadth_first_order(adjacency, 1);
bfs_order # vertex IDs in queue-removal order; these are not distances

Processing vertex 1 adds vertices 2 and 3 to the queue. Processing those vertices adds 4 and 5, and processing vertex 4 adds 6. Taking vertices from the front of the queue gives `[1, 2, 3, 4, 5, 6]`.

Both traversals reach all six vertices, but their visit orders differ. In Task 3, we will compare these orders and test each implementation.

___


## Task 3: Test and compare the traversals

In this task, we will test both implementations and compare the order in which they visit reachable vertices.

**Is a traversal order a path?** A directed path must follow an edge from each vertex to the next. The breadth-first result lists vertex 4 immediately after vertex 3, but the graph has no edge $3\rightarrow4$. The sequence records the order of visits; consecutive entries need not be joined by a directed edge.

Use the table below to identify the positions where the orders differ. Explain why the depth-first search visits vertex 5 before vertex 4, while the breadth-first search visits vertex 4 before vertex 5.


In [ ]:
# Compare visit positions, not edges along a path -
comparison_table = DataFrame(
    visit_position = collect(eachindex(dfs_order)),
    dfs_vertex = dfs_order,
    bfs_vertex = bfs_order,
    same_vertex = dfs_order .== bfs_order, # compare each position, not equality of the reachable sets
);
pretty_table(comparison_table)

**What changes if we start at vertex 3?** Before running the next cell, use the graph to predict which vertices are reachable and whether DFS and BFS will visit them in the same order. Then compute both traversals:


In [ ]:
# Compare reachability and visit order from vertex 3 -
# Each call creates a new visited set; the earlier searches do not carry over.
dfs_from_three = L4bTraversal.depth_first_order(adjacency, 3);
bfs_from_three = L4bTraversal.breadth_first_order(adjacency, 3);
(depth_first = dfs_from_three, breadth_first = bfs_from_three)

Only the chain `3 → 5 → 4 → 6` is reachable, so both traversals should return `[3, 5, 4, 6]`. The edge directions prevent a return to vertices 1 or 2, and there are no competing branches to produce different visit orders.


**What happens when the graph contains a cycle?** We construct the cycle $1\rightarrow2\rightarrow3\rightarrow1$ with an additional edge $2\rightarrow4$. When either traversal encounters vertex 1 again, the visited set prevents it from exploring that vertex again. Both traversals should finish with the order `[1, 2, 3, 4]`.

Use the tests below as the completion check for both functions in [`src/Compute.jl`](src/Compute.jl).


In [ ]:
# Build a cyclic test graph; the helper removes the repeated (1, 2) edge -
cyclic_adjacency = L4bTraversal.adjacency_from_edges([
    (1, 2), (1, 2), (2, 3), (3, 1), (2, 4),
]);
# A shallow dictionary copy would share its neighbor vectors and could hide
# an in-place change. deepcopy gives the check an independent reference.
cyclic_snapshot = deepcopy(cyclic_adjacency);

@testset "deterministic DFS and BFS contracts" begin
    # Check traversal orders from both starting vertices -
    @test dfs_order == [1, 2, 3, 5, 4, 6]
    @test bfs_order == [1, 2, 3, 4, 5, 6]
    @test dfs_from_three == [3, 5, 4, 6]
    @test bfs_from_three == [3, 5, 4, 6]

    # The cycle returns to vertex 1; its visited mark must prevent another exploration.
    @test L4bTraversal.depth_first_order(cyclic_adjacency, 1) == [1, 2, 3, 4]
    @test L4bTraversal.breadth_first_order(cyclic_adjacency, 1) == [1, 2, 3, 4]

    # Check that neither search changed the supplied neighbor lists -
    @test cyclic_adjacency == cyclic_snapshot

    # Reject missing keys and Boolean starts; Bool is an Integer subtype in Julia.
    @test_throws ArgumentError L4bTraversal.depth_first_order(adjacency, 99)
    @test_throws ArgumentError L4bTraversal.depth_first_order(adjacency, true)
    @test_throws ArgumentError L4bTraversal.breadth_first_order(adjacency, true)
    @test_throws ArgumentError L4bTraversal.breadth_first_order(adjacency, 99)
end;

If all tests pass, both implementations produce the expected orders for these graphs, handle the cycle without repeated visits, preserve the tested adjacency list, and reject invalid starting vertices. We have checked both the traversal results and the input requirements.

___


## Summary

We built an adjacency list for a directed graph, implemented two traversals, and compared how they explore the vertices reachable from a chosen start.

> __Key Takeaways:__
>
> * **Adjacency lists and reachability:** We built and checked outgoing-neighbor lists for the directed graph, including an empty list for a vertex with no outgoing edges. Both searches used these lists to find reachable vertices; edge weights did not affect the traversal.
> * **Recursion, queues, and visited sets:** We used recursive calls to follow one branch in depth-first search and [`MyQueue`](../../../code/src/StacksQueues.jl) to explore layers in breadth-first search. We tracked visited vertices to avoid repeated exploration; marking each vertex when enqueued prevented duplicate entries in the breadth-first queue.
> * **Interpreting traversal results:** We distinguished which vertices are reachable from the order in which they are visited. For the same graph and starting vertex, both searches reach the same vertices, but their exploration rules can produce different visit orders. These orders record the search process and need not form a directed path.

In [the L4c lecture](../L4c/CHEME-5800-L4c-Lecture-ShortestPathAlgorithms-Fall-2026.ipynb), we include edge weights and ask which route has the smallest total cost.

___